# Building a Basic RAG Pipeline with Pinecone + LiteLLM

In this notebook we'll build a simple Retrieval-Augmented Generation (RAG) system from scratch:

1. **Create** a Pinecone vector index and load it with sample data
2. **Retrieve** relevant chunks for a user's question — two ways: letting Pinecone embed for us, and embedding the query ourselves
3. **Generate** a grounded answer by passing that context to an LLM
4. **Wrap it up** in reusable functions and a simple Gradio UI

## Setup
Install dependencies, load our API keys from `.env`, and initialize the Pinecone client.

In [ ]:
%pip install -q python-dotenv litellm pinecone gradio

In [ ]:
from dotenv import load_dotenv
from litellm import completion
from pinecone import Pinecone
import gradio as gr
import os

In [ ]:
load_dotenv()
print("Anthropic API key loaded:", os.getenv("ANTHROPIC_API_KEY") is not None) # platform.claude.com
print("Pinecone API key loaded:", os.getenv("PINECONE_API_KEY") is not None) #app.pinecone.io
pc_api = os.environ.get("PINECONE_API_KEY")

In [ ]:
# Initialize Pinecone client
pc = Pinecone(api_key=pc_api)

## Create a Pinecone Index

Create a serverless index that uses Pinecone's **integrated embedding model** (`llama-text-embed-v2`).

In [ ]:
index_name = "monopoly-rules"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

## Load Sample Data into the Index

We'll upsert a small knowledge base of the Game of Monopoly Rules

In [ ]:
"""
Monopoly rules chunked for RAG.

Source: Official Hasbro Monopoly rules (https://www.hasbro.com/common/instruct/00009.pdf)
Note: This is a paraphrased, original-wording restatement of the rules (not a verbatim
copy of Hasbro's text)
"""

MONOPOLY_RULES_CHUNKS = [
    # --- setup ---
    {"id": "1", "chunk_text": "Each player chooses one token to represent them on the board and begins at the GO space.", "category": "setup"},
    {"id": "2", "chunk_text": "Each player starts the game with $1,500 in Monopoly money, distributed as: two $500s, two $100s, two $50s, six $20s, five $10s, five $5s, and five $1s.", "category": "setup"},
    {"id": "3", "chunk_text": "Before play begins, the Chance and Community Chest cards are shuffled separately and placed face-down on their marked spaces on the board.", "category": "setup"},
    {"id": "4", "chunk_text": "One player is chosen to act as the Banker, who also typically serves as the Auctioneer. If the Banker also plays the game, they must keep their personal money completely separate from the Bank's money.", "category": "setup"},
    {"id": "5", "chunk_text": "The Bank holds all money, Title Deed cards, houses, and hotels that have not yet been purchased by players. The Bank pays salaries, sells and auctions property, sells houses and hotels, and collects taxes, fines, and loan interest.", "category": "setup"},
    {"id": "6", "chunk_text": "The Bank can never go bankrupt. If the Bank runs out of physical money, the Banker may write amounts on ordinary paper to serve as additional currency.", "category": "setup"},

    # --- gameplay ---
    {"id": "7", "chunk_text": "Players take turns rolling two dice. The player with the highest total starts the game, and play proceeds clockwise around the table.", "category": "gameplay"},
    {"id": "8", "chunk_text": "On a turn, a player rolls both dice and moves their token clockwise around the board the number of spaces shown on the dice.", "category": "gameplay"},
    {"id": "9", "chunk_text": "Landing on a space may require a player to buy property, pay rent, pay a tax, draw a Chance or Community Chest card, or go to Jail, depending on which space they land on.", "category": "gameplay"},
    {"id": "10", "chunk_text": "If a player rolls doubles, they move their token as normal, resolve the space they land on, and then roll again to take an additional move in the same turn.", "category": "gameplay"},
    {"id": "11", "chunk_text": "If a player rolls doubles three times in a row during the same turn, they must move directly to Jail instead of taking their third move.", "category": "gameplay"},
    {"id": "12", "chunk_text": "Multiple player tokens may occupy the same space on the board at the same time; there is no limit on how many tokens can share a space.", "category": "gameplay"},
    {"id": "13", "chunk_text": "Whenever a player's token lands on or passes over the GO space, the Banker pays that player a $200 salary. This salary is only awarded once per full trip around the board.", "category": "gameplay"},

    # --- buying_property ---
    {"id": "14", "chunk_text": "When a player lands on an unowned property, they may buy it from the Bank at the price printed on the board.", "category": "buying_property"},
    {"id": "15", "chunk_text": "A player who buys a property receives the matching Title Deed card, which they place face-up in front of themselves as proof of ownership.", "category": "buying_property"},
    {"id": "16", "chunk_text": "If a player declines to buy the property they landed on, the Banker immediately puts that property up for auction to the highest bidder among all players.", "category": "buying_property"},
    {"id": "17", "chunk_text": "In a property auction, any player may bid, including the player who originally declined to buy the property at its printed price. Bidding may start at any amount, and the winning bidder pays their bid amount to the Bank in exchange for the Title Deed.", "category": "buying_property"},

    # --- rent ---
    {"id": "18", "chunk_text": "When a player lands on a property owned by another player, they must pay that owner rent according to the amount listed on the property's Title Deed card.", "category": "rent"},
    {"id": "19", "chunk_text": "No rent can be collected on a property that is currently mortgaged. A mortgaged property's Title Deed card is kept face-down in front of its owner as a reminder.", "category": "rent"},
    {"id": "20", "chunk_text": "If a player owns every property in a color group, they may charge double the normal rent on any unimproved (no houses or hotels) properties within that group.", "category": "rent"},
    {"id": "21", "chunk_text": "The double-rent bonus for owning a complete color group still applies to unmortgaged properties even if another property in that same color group is mortgaged.", "category": "rent"},
    {"id": "22", "chunk_text": "A property owner forfeits their right to collect rent from a player if they fail to request payment before the second player after that has taken their turn.", "category": "rent"},

    # --- chance_community_chest ---
    {"id": "23", "chunk_text": "When a player lands on a Chance or Community Chest space, they draw the top card from the corresponding deck, follow its instructions, and then return it face-down to the bottom of that deck.", "category": "chance_community_chest"},
    {"id": "24", "chunk_text": "A 'Get Out of Jail Free' card can be kept by the player who draws it until they choose to use it, rather than being returned to the deck immediately.", "category": "chance_community_chest"},
    {"id": "25", "chunk_text": "A player holding a 'Get Out of Jail Free' card may sell it to another player at any time, for any price both players agree to.", "category": "chance_community_chest"},

    # --- income_tax ---
    {"id": "26", "chunk_text": "A player who lands on the Income Tax space must choose between two payment options: pay a flat $200 to the Bank, or pay 10% of their total net worth to the Bank.", "category": "income_tax"},
    {"id": "27", "chunk_text": "For Income Tax purposes, a player's total net worth includes all cash on hand plus the printed prices of all owned properties (mortgaged or not) plus the purchase cost of all houses and hotels they own.", "category": "income_tax"},
    {"id": "28", "chunk_text": "A player must choose their Income Tax payment option (flat fee vs. percentage of net worth) before calculating their total net worth, not after.", "category": "income_tax"},

    # --- jail ---
    {"id": "29", "chunk_text": "A player is sent to Jail in three situations: landing directly on the 'Go to Jail' space, drawing a card that instructs them to go to Jail, or rolling doubles three times in a row on a single turn.", "category": "jail"},
    {"id": "30", "chunk_text": "A player sent to Jail does not collect their $200 GO salary for that move, and their turn ends immediately once their token is placed in Jail.", "category": "jail"},
    {"id": "31", "chunk_text": "A player who simply lands on the Jail space during normal movement, without being sent there, is 'Just Visiting' and suffers no penalty; they continue playing normally on their next turn.", "category": "jail"},
    {"id": "32", "chunk_text": "A player can get out of Jail by rolling doubles on any of their next three turns; if successful, they immediately move forward the number of spaces shown by that doubles roll (without an extra turn).", "category": "jail"},
    {"id": "33", "chunk_text": "A player can get out of Jail by using a 'Get Out of Jail Free' card, either one they already hold or one purchased from another player.", "category": "jail"},
    {"id": "34", "chunk_text": "A player can get out of Jail by paying a $50 fine to the Bank before rolling the dice, on either of their first two turns in Jail.", "category": "jail"},
    {"id": "35", "chunk_text": "If a player fails to roll doubles by their third turn in Jail, they must pay the $50 fine, then immediately move forward the number of spaces shown by that turn's dice roll.", "category": "jail"},
    {"id": "36", "chunk_text": "A player who is in Jail may still buy and sell property, buy and sell houses and hotels, and collect rent from other players as normal.", "category": "jail"},

    # --- free_parking ---
    {"id": "37", "chunk_text": "A player who lands on Free Parking receives no money, property, or reward of any kind; it is simply a resting space with no game effect.", "category": "free_parking"},

    # --- houses ---
    {"id": "38", "chunk_text": "A player may only buy houses from the Bank for properties in a color group if they own every property in that entire color group.", "category": "houses"},
    {"id": "39", "chunk_text": "The price of each house is shown on the Title Deed card of the specific property where it is being built, and can vary between color groups.", "category": "houses"},
    {"id": "40", "chunk_text": "Houses must be built evenly across a color group: a player cannot build a second house on any property in the group until every property in that group has at least one house.", "category": "houses"},
    {"id": "41", "chunk_text": "Once every property in a color group has one house, a player may begin adding a second row of houses, continuing evenly up to a maximum of four houses per property.", "category": "houses"},
    {"id": "42", "chunk_text": "A property owner who has built houses on a color group still collects double rent on any unimproved properties within that same group.", "category": "houses"},

    # --- hotels ---
    {"id": "43", "chunk_text": "A player may buy a hotel for a property once they have four houses on every property in that color group.", "category": "hotels"},
    {"id": "44", "chunk_text": "To build a hotel, a player returns the four houses from that property to the Bank and pays the hotel price listed on the Title Deed card.", "category": "hotels"},
    {"id": "45", "chunk_text": "Only one hotel can be built on any single property, regardless of color group size.", "category": "hotels"},

    # --- building_shortages ---
    {"id": "46", "chunk_text": "If the Bank has no houses left to sell, players who want to build must wait until other players sell or return houses to the Bank before they can build more.", "category": "building_shortages"},
    {"id": "47", "chunk_text": "If there is a limited supply of houses or hotels and multiple players want to buy more than the Bank has available, the Bank auctions the remaining houses or hotels to the highest bidder.", "category": "building_shortages"},

    # --- selling_property ---
    {"id": "48", "chunk_text": "Unimproved properties, railroads, and utilities can be sold directly to another player as a private deal for any price both players agree on.", "category": "selling_property"},
    {"id": "49", "chunk_text": "A property cannot be sold to another player while any property in its color group still has houses or hotels built on it; those buildings must be sold back to the Bank first.", "category": "selling_property"},
    {"id": "50", "chunk_text": "Houses and hotels can be sold back to the Bank at any time for half the price the player originally paid for them.", "category": "selling_property"},
    {"id": "51", "chunk_text": "When selling houses back to the Bank, they must be sold off evenly across a color group, in the reverse order from how they were originally built.", "category": "selling_property"},

    # --- mortgages ---
    {"id": "52", "chunk_text": "Unimproved properties can be mortgaged with the Bank at any time in exchange for the mortgage value printed on the property's Title Deed card.", "category": "mortgages"},
    {"id": "53", "chunk_text": "Before an improved property can be mortgaged, all houses and hotels on every property in its color group must first be sold back to the Bank at half price.", "category": "mortgages"},
    {"id": "54", "chunk_text": "No rent can be collected on a mortgaged property, but rent can still be collected normally on unmortgaged properties within the same color group.", "category": "mortgages"},
    {"id": "55", "chunk_text": "To lift a mortgage, the property's owner must repay the Bank the full mortgage amount plus 10% interest.", "category": "mortgages"},
    {"id": "56", "chunk_text": "A player who buys a mortgaged property from another player may immediately pay off the mortgage plus 10% interest; if they don't, they still owe the Bank 10% interest at the time of purchase, and must pay the interest again if they lift the mortgage later.", "category": "mortgages"},

    # --- bankruptcy ---
    {"id": "57", "chunk_text": "A player is declared bankrupt when they owe more money than they can pay to either another player or the Bank.", "category": "bankruptcy"},
    {"id": "58", "chunk_text": "If a bankrupt player owes another player, they must turn over all their remaining assets of value to that player and leave the game.", "category": "bankruptcy"},
    {"id": "59", "chunk_text": "When settling a debt to another player, any houses or hotels owned by the bankrupt player are sold back to the Bank for half price, and that cash goes to the creditor.", "category": "bankruptcy"},
    {"id": "60", "chunk_text": "If a player owes the Bank more than they can pay even after selling buildings and mortgaging property, they must surrender all assets to the Bank, which then auctions off the properties (but not buildings) and removes the player from the game. The last remaining player wins.", "category": "bankruptcy"},

    # --- speed_die (optional variant) ---
    {"id": "61", "chunk_text": "The Speed Die is an optional third die that can be added to speed up gameplay; it is not part of the classic rules and is only used once players opt into that variant.", "category": "speed_die"},
    {"id": "62", "chunk_text": "When playing with the Speed Die, each player receives an extra $1,000 at the start of the game in addition to the standard $1,500.", "category": "speed_die"},
    {"id": "63", "chunk_text": "In Speed Die play, a player does not use the Speed Die until after they have landed on or passed GO for the first time and collected their initial $200 salary.", "category": "speed_die"},
    {"id": "64", "chunk_text": "Once a player starts using the Speed Die, they roll it together with the two normal dice each turn: a result of 1, 2, or 3 is simply added to the normal dice total.", "category": "speed_die"},
    {"id": "65", "chunk_text": "Rolling the 'Bus' symbol on the Speed Die lets a player move using just one of the two normal dice values, or their sum, whichever they choose.", "category": "speed_die"},
    {"id": "66", "chunk_text": "Rolling the 'Mr. Monopoly' symbol on the Speed Die moves the player forward by the normal dice total, then advances them either to the next unowned Bank property (which they may buy or send to auction) or to the next property on which they owe rent, depending on whether the Bank still holds any properties.", "category": "speed_die"},
    {"id": "67", "chunk_text": "In Speed Die play, only the two normal dice are checked for doubles; the Speed Die itself is ignored for that purpose, and rolling three-of-a-kind across all three dice lets a player move to any space on the board.", "category": "speed_die"},

    # --- misc ---
    {"id": "68", "chunk_text": "Players may only borrow money from the Bank, and only by mortgaging property; players are not allowed to lend money to or borrow money from each other directly.", "category": "misc"},
    {"id": "69", "chunk_text": "The overall objective of Monopoly is to become the wealthiest player by buying, renting out, and selling property, ultimately driving all other players into bankruptcy.", "category": "misc"},
]

In [ ]:
index = pc.Index(index_name)
index.upsert_records(
    namespace="default",
    records=MONOPOLY_RULES_CHUNKS
)

In [ ]:
stats = index.describe_index_stats()
print(stats)

## Retrieval-Augmented Generation (RAG)

RAG: retrieve relevant chunks from our source and shove them into our LLM Prompt to help provide response.

There are two ways to retrieve from Pinecone:

1. **Integrated embeddings** — pass raw text, Pinecone embeds it for you
2. **Bring your own embeddings** — embed the query yourself, then search with the resulting vector

We'll walk through both below.

### Option 1: Search with Integrated Embeddings

Since our index uses Pinecone's hosted embedding model, we can search directly with plain text — Pinecone embeds the query behind the scenes and returns the closest matches.

In [ ]:
# semantic search - Pinecone uses the embedding model integrated with the index to convert the text to a dense vector automatically.
results = index.search(
    namespace="default", 
    query={
        "inputs": {"text": "what happens when a player is sent to jail"}, 
        "top_k": 3
    },
    fields=["category", "chunk_text"]
)

In [ ]:
for hit in results.result.hits:
  print(f"Score: {hit.score}")
  print(f"Category: {hit.fields['category']}")
  print(f"Text: {hit.fields['chunk_text']}")
  print("-" * 80)

### Option 2: Search with Your Own Embeddings

If you want to control the embedding model yourself (or your index isn't integrated), embed the query first, then pass the resulting vector to `index.query()`.

- First convert the query to a vector (using whatever embedding model matches your index), then plug in the vector
- https://docs.pinecone.io/guides/search/semantic-search?retry=2

*Note: the cell below uses a placeholder vector and namespace purely to show the `index.query()` syntax — it isn't part of our working example. The next cell embeds a real query and searches our actual index.*

In [ ]:
# Manually embed the query text
query_embedding = pc.inference.embed(
    model="llama-text-embed-v2",
    inputs=["what happens when a player is sent to jail"],
    parameters={"input_type": "query", "truncate": "END"}
)

print("Query embedding length:", len(query_embedding[0].values))
print("Query embedding:", query_embedding[0].values[:5],)

Now search our index using the real query embedding computed above:

In [ ]:

# Use the resulting vector to search the index
results2 = index.query(
    namespace="default", 
    vector=query_embedding[0].values,
    top_k=3,
    include_metadata=True
)
print(results2)

In [ ]:
for match in results2.matches:
    print(f"Score: {match.score:}")
    print(f"Chunk: {match.metadata['chunk_text']}")
    print(f"Category: {match.metadata.get('category')}")
    print("---")

### Build the Context String

Extract the text from each retrieved match and join them into a single context block to feed the LLM.

Option 1:

In [ ]:
context_chunks = []
for hit in results["result"]["hits"]:
    text = hit["fields"]["chunk_text"]
    context_chunks.append(f" {text}")

context = "\n\n".join(context_chunks)
print(context)

Option 2:

In [ ]:
context_chunks = []
for match in results2["matches"]:
    text = match["metadata"].get("chunk_text", "")
    context_chunks.append(text)

context2 = "\n\n".join(context_chunks)

print(context2)

### Generate an Answer with the Retrieved Context

Shove the context into your LLM prompt

In [ ]:
response = completion(
    model="anthropic/claude-haiku-4-5",
    messages=[
        {
            "role": "system",
            "content": f"Use the following context to answer the user's question:\n\n{context2}"
        },
        {
            "role": "user",
            "content": "what happens when a player is sent to jail?"
        }
    ]
)

In [ ]:
print(response["choices"][0]["message"]["content"])

## Wrap It Up: Reusable RAG Functions

### Version 1 — Integrated Embedding

The same two-step pattern, but using Pinecone's integrated embeddings (`index.search()`) so we never call an embedding model directly.

In [ ]:
def retrieve_context1(query: str, index, top_k: int) -> str:
    """
    Embeds the query and retrieves the top-k most relevant chunks from Pinecone.
    Returns a single concatenated context string.
    """
    # semantic search - Pinecone uses the embedding model integrated with the index to convert the text to a dense vector automatically.
    results = index.search(
        namespace="default", 
        query={
            "inputs": {"text": query}, 
            "top_k": top_k
        },
        fields=["category", "chunk_text"]
    )

    # chunks = [match["metadata"].get("text", "") for match in results["matches"]]
    context_chunks = []
    for hit in results["result"]["hits"]:
        text = hit["fields"]["chunk_text"]
        context_chunks.append(f" {text}")

    context = "\n\n".join(context_chunks)
    return context


def retrieve_and_generate1(
    prompt: str,
    index: str,
    model: str,
    top_k: int
) -> str:
    """
    Retrieves relevant context from Pinecone, then generates a response
    using the LLM with that context injected as a system message.
    """
    context = retrieve_context1(prompt, index, top_k=top_k)

    response = completion(
        model=model,
        messages=[
            {
                "role": "system",
                "content": f"Use the following context to answer the user's question:\n\n{context}"
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["choices"][0]["message"]["content"]

Try it out:

In [ ]:
prompt = "what happens when a player is sent to jail?"
index = pc.Index("monopoly-rules")
model = "anthropic/claude-haiku-4-5"
top_k = 3

answer = retrieve_and_generate1(prompt, index, model=model, top_k=top_k)
print(answer)

### Version 2 — Manual Embedding
Combines the "bring your own embeddings" steps above (embed → search → build context → generate) into two reusable functions: `retrieve_context()` and `retrieve_and_generate()`.

In [ ]:
def retrieve_context2(prompt: str, index, top_k: int, embedding_model: str) -> str:
    """
    Embeds the query and retrieves the top-k most relevant chunks from Pinecone.
    Returns a single concatenated context string.
    """
    
    query_embedding = pc.inference.embed(
        model=embedding_model,
        inputs=[prompt],
        parameters={"input_type": "query", "truncate": "END"}
    )

    results2 = index.query(
        namespace="default", 
        vector=query_embedding[0].values,
        top_k=top_k,
        include_metadata=True
    )

    context_chunks = []
    for match in results2["matches"]:
        text = match["metadata"].get("chunk_text", "")
        context_chunks.append(text)

    context2 = "\n\n".join(context_chunks)
    return context2


def retrieve_and_generate2(
    prompt: str,
    index,
    embedding_model: str,
    inference_model: str,
    top_k: int
) -> str:
    """
    Retrieves relevant context from Pinecone, then generates a response
    using the LLM with that context injected as a system message.
    """
    context = retrieve_context2(prompt, index, top_k=top_k, embedding_model=embedding_model)

    response = completion(
        model=inference_model,
        messages=[
            {
                "role": "system",
                "content": f"Use the following context to answer the user's question:\n\n{context}"
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["choices"][0]["message"]["content"]

In [ ]:
prompt = "what happens if I go to jail?"
index = pc.Index("monopoly-rules")
embedding_model = "llama-text-embed-v2"
inference_model = "anthropic/claude-haiku-4-5"
top_k = 3

answer = retrieve_and_generate2(prompt, index, embedding_model=embedding_model, inference_model=inference_model, top_k=top_k)
print(answer)

## Interactive UI with Gradio

Finally, let's wrap `retrieve_and_generate2()` in a simple Gradio interface so we can ask questions through a text box instead of editing code.

In [ ]:
def respond(message: str):
    answer = retrieve_and_generate2(
        message,
        index,
        embedding_model="llama-text-embed-v2",
        inference_model="anthropic/claude-haiku-4-5",
        top_k=3,
    )
    return answer


with gr.Blocks(title="Monopoly Rules Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎲 Monopoly Rules Chatbot")
    gr.Markdown("Ask a question about Monopoly rules and get an answer grounded in the retrieved context.")

    msg = gr.Textbox(label="Question", placeholder="e.g. How do I start a game of Monopoly?")
    submit_btn = gr.Button("Submit", variant="primary")

    with gr.Group():
        answer = gr.Markdown(label="Answer")

    gr.Examples(
        examples=[
            "How do I start a game of Monopoly?",
            "What happens if I land on Free Parking?",
            "How do I get out of Jail?",
        ],
        inputs=msg,
    )

    msg.submit(respond, inputs=msg, outputs=answer)
    submit_btn.click(respond, inputs=msg, outputs=answer)

demo.launch()

In [ ]:
demo.close()